# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints based on their documentation.

In [ ]:
# The correct base URL according to https://footballdata.io/documentation/endpoints/
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    # Standard Authorization approaches.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Test Connection (Leagues)
Verify the API key works and view the generic payload structure.

In [ ]:
test_endpoint = "leagues"

print(f"Fetching endpoint: /{test_endpoint}...")
data = fetch_footballdata(test_endpoint)

if data:
    print("\n✅ Success! Here is a snippet of the response:\n")
    print(json.dumps(data, indent=2)[:500] + "\n...[truncated]")
else:
    print("\n⚠️ Failed to retrieve data.")

## 3. Fetching Match Endpoints
The V4 pipeline needs live events, starting XIs, and advanced match stats. Let's pull `fixtures/today` to grab a valid `match_id` for testing.

In [ ]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")
match_id_to_test = None

if today_data and today_data.get("success") and today_data.get("data"):
    # Grab the first match ID from the list
    match_id_to_test = today_data["data"][0].get("match_id")
    print(f"\n✅ Found matches today! We will use match_id: {match_id_to_test} for detailed tests.")
else:
    print("\n⚠️ No fixtures found today or API response was invalid.")
    # Fallback to a known match if available, or just leave None
    match_id_to_test = 1000 # Dummy ID just to show the structure

## 4. Fetching Granular Match Data (Stats & Events)
We need to fetch the stats and events endpoints for our model calculations (Likelihood updates & Live Posteriors).

In [ ]:
if match_id_to_test:
    # 1. Fetch Stats
    print(f"Fetching stats for match {match_id_to_test}...")
    stats_data = fetch_footballdata(f"matches/{match_id_to_test}/stats")
    if stats_data:
        print("\n📊 Stats JSON Structure:\n", json.dumps(stats_data, indent=2)[:500] + "\n...[truncated]")
        
    # 2. Fetch Events
    print(f"\nFetching events for match {match_id_to_test}...")
    events_data = fetch_footballdata(f"matches/{match_id_to_test}/events")
    if events_data:
        print("\n⏱️ Events JSON Structure:\n", json.dumps(events_data, indent=2)[:500] + "\n...[truncated]")
else:
    print("No match_id available to test detailed endpoints.")

## 5. Build the V4 Ingestion Parser
This function will accept the raw Footballdata.io JSON payloads and transform them into the uniform dictionary format expected by `v4_backend/likelihood_adjustment.py` and `v4_backend/in_play_posterior.py`.

In [ ]:
def parse_footballdata_to_v4(match_info, events_payload, stats_payload):
    """
    Parses Footballdata.io responses into the V4 standard schema.
    """
    # Note: These keys will need to be perfectly mapped once you inspect the exact output above!
    parsed = {
        "home_team": match_info.get("home_team_name"),
        "away_team": match_info.get("away_team_name"),
        "home_score": match_info.get("home_score", 0),
        "away_score": match_info.get("away_score", 0),
        "current_minute": match_info.get("minute", 0),
        "red_cards": {"home": 0, "away": 0},
        "live_xg": {"home": 0.0, "away": 0.0},
        "starting_xi": {"home": [], "away": []}
    }
    
    # Process Stats (e.g., live xG)
    if stats_payload and stats_payload.get("success"):
        # TBD: Map the exact nested path for expected goals
        pass 
        
    # Process Events (e.g., Red Cards)
    if events_payload and events_payload.get("success"):
        for event in events_payload.get("data", []):
            if event.get("type") == "Red Card":
                if event.get("team") == "home":
                    parsed["red_cards"]["home"] += 1
                else:
                    parsed["red_cards"]["away"] += 1
                    
    return parsed